# ReCAHS — Reproducibility Setup and Audit

Bu notebook, projenin yeniden üretilebilirlik ortamını ve denetim kayıtlarını tek seferde hazırlar.

- Google Drive'ı bağlar.
- Projeyi doğru GitHub branch'inden indirir.
- ETTh1 verisini ve mevcut B4 checkpoint'ini bulur.
- Veri/checkpoint SHA256 değerlerini hesaplar.
- Python, PyTorch, CUDA ve paket sürümlerini kaydeder.
- Time-Series-Library commit'ini kaydeder.
- `configs/`, `src/` ve `scripts/` başlangıç dosyalarını oluşturur.
- Sonuçları Drive'a JSON raporu olarak yazar.

> Bu notebook eğitim başlatmaz ve mevcut checkpoint'in üzerine yazmaz.

In [ ]:
# 1) AYARLAR — Gerekirse yalnızca PROJECT_DIR değerini değiştir.
from pathlib import Path

REPO_URL = 'https://github.com/didemneda/regime-aware-head-pruning.git'
BASE_BRANCH = 'didem/patchtst-baseline'
WORK_BRANCH = 'feat/reproducible-pipeline'
REPO_DIR = Path('/content/regime-aware-head-pruning')
TSLIB_DIR = Path('/content/Time-Series-Library')
PROJECT_DIR = Path('/content/drive/MyDrive/BIL401_Regime_Head_Pruning')

# Daha sonra sabitleyeceğiz. Şimdilik None bırakılırsa indirilen commit rapora yazılır.
TSLIB_COMMIT = None

print('Repo:', REPO_URL)
print('Çalışma branch:', WORK_BRANCH)
print('Drive proje klasörü:', PROJECT_DIR)

In [ ]:
# 2) GOOGLE DRIVE'I BAĞLA
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print('Drive hazır:', PROJECT_DIR)

In [ ]:
# 3) REPOYU İNDİR VE ÇALIŞMA BRANCH'İNİ HAZIRLA
# Branch GitHub'da varsa onu kullanır; yoksa BASE_BRANCH'ten yerel olarak oluşturur.
import subprocess

def run(command, cwd=None, check=True):
    print('$', ' '.join(map(str, command)))
    result = subprocess.run(
        list(map(str, command)),
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True,
    )
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print(result.stderr.strip())
    if check and result.returncode != 0:
        raise RuntimeError(f'Komut başarısız (kod={result.returncode}): {command}')
    return result

if not (REPO_DIR / '.git').exists():
    remote_check = run(
        ['git', 'ls-remote', '--heads', REPO_URL, WORK_BRANCH],
        check=False,
    )
    remote_has_work_branch = bool(remote_check.stdout.strip())
    clone_branch = WORK_BRANCH if remote_has_work_branch else BASE_BRANCH
    run(['git', 'clone', '--branch', clone_branch, REPO_URL, REPO_DIR])
else:
    run(['git', 'fetch', 'origin'], cwd=REPO_DIR)
    remote_check = run(
        ['git', 'ls-remote', '--heads', 'origin', WORK_BRANCH],
        cwd=REPO_DIR,
        check=False,
    )
    remote_has_work_branch = bool(remote_check.stdout.strip())

local_check = run(
    ['git', 'show-ref', '--verify', '--quiet', f'refs/heads/{WORK_BRANCH}'],
    cwd=REPO_DIR,
    check=False,
)

if local_check.returncode == 0:
    run(['git', 'switch', WORK_BRANCH], cwd=REPO_DIR)
elif remote_has_work_branch:
    run(['git', 'switch', '--track', f'origin/{WORK_BRANCH}'], cwd=REPO_DIR)
else:
    run(['git', 'switch', '-c', WORK_BRANCH], cwd=REPO_DIR)
    print('NOT: Branch yerel olarak oluşturuldu. Push işlemini daha sonra yapacağız.')

repo_commit = run(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR).stdout.strip()
repo_branch = run(['git', 'branch', '--show-current'], cwd=REPO_DIR).stdout.strip()
print('Aktif branch:', repo_branch)
print('Repo commit:', repo_commit)

In [ ]:
# 4) DATASET VE CHECKPOINT'İ BUL, SHA256 HESAPLA
import hashlib

def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as file:
        for block in iter(lambda: file.read(block_size), b''):
            digest.update(block)
    return digest.hexdigest()

dataset_candidates = [
    PROJECT_DIR / 'data' / 'ETTh1.csv',
    TSLIB_DIR / 'dataset' / 'ETDataset' / 'ETT-small' / 'ETTh1.csv',
]
DATASET_PATH = next((p for p in dataset_candidates if p.exists()), None)

checkpoint_candidates = sorted(PROJECT_DIR.glob(
    'checkpoints/B4_patchtst_etth1_336_dm128_h8/**/checkpoint.pth'
))

# Bazı kayıtlarda checkpoint doğrudan ana klasörde olabilir.
direct_checkpoint = (
    PROJECT_DIR / 'checkpoints' / 'B4_patchtst_etth1_336_dm128_h8' / 'checkpoint.pth'
)
if direct_checkpoint.exists() and direct_checkpoint not in checkpoint_candidates:
    checkpoint_candidates.insert(0, direct_checkpoint)

CHECKPOINT_PATH = checkpoint_candidates[0] if len(checkpoint_candidates) == 1 else None

print('Project folder:', PROJECT_DIR.exists())
print('Dataset:', DATASET_PATH)
print('Checkpoint aday sayısı:', len(checkpoint_candidates))
for candidate in checkpoint_candidates:
    print(' -', candidate)

dataset_sha256 = sha256_file(DATASET_PATH) if DATASET_PATH else None
checkpoint_sha256 = sha256_file(CHECKPOINT_PATH) if CHECKPOINT_PATH else None

if DATASET_PATH:
    print('ETTh1 SHA256:', dataset_sha256)
else:
    print('UYARI: ETTh1.csv bulunamadı.')

if CHECKPOINT_PATH:
    size_mb = CHECKPOINT_PATH.stat().st_size / 1024**2
    print('Checkpoint:', CHECKPOINT_PATH)
    print('Checkpoint boyutu (MB):', round(size_mb, 2))
    print('Checkpoint SHA256:', checkpoint_sha256)
elif len(checkpoint_candidates) == 0:
    print('UYARI: B4 checkpoint bulunamadı. Eğitim başlatılmadı.')
else:
    print('UYARI: Birden fazla checkpoint bulundu; otomatik seçim yapılmadı.')

In [ ]:
# 5) TIME-SERIES-LIBRARY'Yİ HAZIRLA VE COMMIT'İ KAYDET
import sys

if not (TSLIB_DIR / '.git').exists():
    run(['git', 'clone', 'https://github.com/thuml/Time-Series-Library.git', TSLIB_DIR])
else:
    run(['git', 'fetch', 'origin'], cwd=TSLIB_DIR)

if TSLIB_COMMIT:
    run(['git', 'checkout', TSLIB_COMMIT], cwd=TSLIB_DIR)

tslib_commit = run(['git', 'rev-parse', 'HEAD'], cwd=TSLIB_DIR).stdout.strip()
tslib_status = run(['git', 'status', '--short'], cwd=TSLIB_DIR).stdout.strip()

if str(TSLIB_DIR) not in sys.path:
    sys.path.insert(0, str(TSLIB_DIR))

print('Time-Series-Library commit:', tslib_commit)
print('Çalışma ağacı temiz:', not bool(tslib_status))

In [ ]:
# 6) ORTAM BİLGİLERİNİ TOPLA
import importlib.metadata as metadata
import os
import platform
import random
import sys

import numpy as np
import torch

TRACKED_PACKAGES = [
    'torch', 'numpy', 'pandas', 'scikit-learn', 'scipy',
    'statsmodels', 'matplotlib', 'PyYAML', 'tqdm', 'thop'
]

def installed_version(package):
    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return None

package_versions = {name: installed_version(name) for name in TRACKED_PACKAGES}
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None

environment = {
    'python': sys.version.replace('\n', ' '),
    'platform': platform.platform(),
    'pytorch': torch.__version__,
    'cuda_runtime': torch.version.cuda,
    'cuda_available': torch.cuda.is_available(),
    'cudnn': torch.backends.cudnn.version() if torch.cuda.is_available() else None,
    'gpu': gpu_name,
    'packages': package_versions,
}

for key, value in environment.items():
    print(f'{key}: {value}')

In [ ]:
# 7) REPRODUCIBILITY İSKELET DOSYALARINI OLUŞTUR
# Var olan dosyaların üzerine yazmaz.
import textwrap

FILES = {
    'configs/etth1.yaml': f'''
dataset:
  name: ETTh1
  file: ETTh1.csv
  sha256: {dataset_sha256 or 'TO_BE_FILLED'}
model:
  name: PatchTST
  seq_len: 336
  label_len: 48
  pred_len: 96
  enc_in: 7
  dec_in: 7
  c_out: 7
  e_layers: 3
  d_layers: 1
  d_model: 128
  d_ff: 256
  n_heads: 8
training:
  batch_size: 32
  epochs: 10
  patience: 3
  learning_rate: 0.0001
  seeds: [7, 42, 1234, 2026, 3407]
external:
  time_series_library_commit: {tslib_commit}
''',
    'src/__init__.py': '',
    'src/reproducibility.py': '''
import os
import random

import numpy as np
import torch


def set_global_seed(seed: int, deterministic: bool = True) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_generator(seed: int) -> torch.Generator:
    generator = torch.Generator()
    generator.manual_seed(seed)
    return generator
''',
    'src/paths.py': '''
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ProjectPaths:
    project_dir: Path
    tslib_dir: Path

    @property
    def data_dir(self) -> Path:
        return self.project_dir / 'data'

    @property
    def checkpoint_dir(self) -> Path:
        return self.project_dir / 'checkpoints'

    @property
    def results_dir(self) -> Path:
        return self.project_dir / 'results'

    def create_output_dirs(self) -> None:
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        self.results_dir.mkdir(parents=True, exist_ok=True)
''',
}

created = []
skipped = []
for relative_path, content in FILES.items():
    target = REPO_DIR / relative_path
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        skipped.append(relative_path)
        continue
    target.write_text(textwrap.dedent(content).lstrip(), encoding='utf-8')
    created.append(relative_path)

print('Oluşturulan dosyalar:')
for path in created:
    print(' +', path)
print('Zaten bulunduğu için atlanan dosyalar:')
for path in skipped:
    print(' =', path)

In [ ]:
# 8) AUDIT RAPORUNU DRIVE'A KAYDET
import json
from datetime import datetime, timezone

audit_dir = PROJECT_DIR / 'reproducibility'
audit_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
audit_path = audit_dir / f'reproducibility_audit_{timestamp}.json'

audit = {
    'created_at_utc': timestamp,
    'repository': {
        'url': REPO_URL,
        'branch': repo_branch,
        'commit': repo_commit,
    },
    'time_series_library': {
        'path': str(TSLIB_DIR),
        'commit': tslib_commit,
        'working_tree_clean': not bool(tslib_status),
    },
    'dataset': {
        'path': str(DATASET_PATH) if DATASET_PATH else None,
        'sha256': dataset_sha256,
    },
    'checkpoint': {
        'path': str(CHECKPOINT_PATH) if CHECKPOINT_PATH else None,
        'sha256': checkpoint_sha256,
        'candidate_count': len(checkpoint_candidates),
    },
    'environment': environment,
    'scaffold': {
        'created': created,
        'skipped': skipped,
    },
}

audit_path.write_text(
    json.dumps(audit, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

print('\nAUDIT TAMAMLANDI')
print('Rapor:', audit_path)
print('Repo:', REPO_DIR)
print('Branch:', repo_branch)
print('Git durumunu görmek için sonraki hücreyi çalıştır.')

In [ ]:
# 9) SON KONTROL — Bu hücre GitHub'a hiçbir şey göndermez.
run(['git', 'status', '--short'], cwd=REPO_DIR)

print('\nBana şu dört bilgiyi gönder:')
print('1. AUDIT TAMAMLANDI satırının altındaki rapor yolu')
print('2. ETTh1 SHA256')
print('3. Time-Series-Library commit')
print('4. Checkpoint bulundu mu ve aday sayısı kaç?')
print('\nHenüz git add/commit/push yapma; önce çıktıları birlikte kontrol edeceğiz.')